# Atividade 1 de Processamento de Imagens
**Aluno(a):** Ana Beatriz de Oliveira Duarte
**Matrícula:** 1758190  
**Tema do Trabalho Final:** Gastronomia 

---

### Resumo das Regras e Diretrizes
1. **Uso Restrito de OpenCV:** Utilizado **exclusivamente** para:
   - Carregamento de imagens (`cv2.imread`);
   - Salvamento de imagens (`cv2.imwrite`);
   - Aplicação do filtro de desfoque gaussiano (`cv2.GaussianBlur`) conforme solicitado no item *(ii)* da Questão 1.
2. **Implementações Manuais (via NumPy):** Todas as transformações de intensidade, combinações lineares, permutações em blocos, inversões espaciais, espelhamentos e quantizações foram desenvolvidas manualmente sem funções prontas não autorizadas.
3. **Relatório Completo Integrado:** O notebook contém formulação matemática detalhada, inspeções visuais comparativas, gráficos de curvas de transferência, mapas de erro, métricas quantitativas (PSNR/RMSE) e discussões técnicas dos efeitos observados.


In [5]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

# Ajustes para visualização no Jupyter Notebook
%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "image.cmap": "gray"
})

# Verificando as versões das bibliotecas
opencv_version = cv2.__version__
numpy_version = np.__version__

print("Versão do OpenCV:", opencv_version)
print("Versão do NumPy:", numpy_version)

Versão do OpenCV: 5.0.0
Versão do NumPy: 2.4.6


In [6]:

"""
essa funcao converte uma imagem na escala BGR para tons de cinza utilizando
os pesos definidos pelo padrão ITU-R BT.601.
"""
def converter_para_cinza(img: np.ndarray) -> np.ndarray:

    if img.ndim != 3 or img.shape[-1] != 3:
        raise ValueError("A imagem precisa ter três canais de cor (BGR).")

    blue = img[:, :, 0].astype(np.float32)
    green = img[:, :, 1].astype(np.float32)
    red = img[:, :, 2].astype(np.float32)

    # cálculo da luminância 
    resultado = (
        red * 0.299 +
        green * 0.587 +
        blue * 0.114
    )

    # mantém os valores dentro do intervalo de uma imagem de 8 bits
    resultado = np.clip(resultado, 0, 255)

    return resultado.astype(np.uint8)

## 1. Questão 1: Efeito de Esboço a Lápis (*Pencil Sketch*)

### 1.1. Enunciado
> **1. Implementar um efeito de esboço a lápis em uma imagem por meio dos seguintes passos:**  
> **(i)** Converter a imagem colorida para níveis de cinza;  
> **(ii)** Aplicar um filtro de desfoque gaussiano do OpenCV (por exemplo, com uma máscara de $21 \times 21$ pixels) para suavizar os detalhes da imagem;  
> **(iii)** Dividir a imagem em tons de cinza pela versão desfocada para realçar os contornos.

---

In [7]:
# pipeline do esboço da questão

"""
    essa funcao realiza as etapas necessárias para produzir o efeito
    de esboço a lápis.
    """
def gerar_esboco(imagem_bgr: np.ndarray,
                 tamanho_kernel=(21, 21),
                 sigma=0) -> tuple:

    imagem_cinza = converter_cinza(imagem_bgr)
    
    imagem_suave = cv2.GaussianBlur(
        imagem_cinza,
        tamanho_kernel,
        sigmaX=sigma,
        sigmaY=sigma
    )

    cinza_float = imagem_cinza.astype(np.float32)
    suave_float = imagem_suave.astype(np.float32)

    pequeno_valor = 1e-5
    
    resultado = (cinza_float / (suave_float + pequeno_valor)) * 255

    esboco = np.clip(resultado, 0, 255).astype(np.uint8)

    return imagem_cinza, imagem_suave, esboco

In [8]:
# funcao para exibir os resultados

"""
    essa funcao abre uma imagem, aplica o efeito de esboço e apresenta
    todas as etapas do processamento.
    """
def mostrar_resultados(caminho_imagem: str, nome_imagem: str, tamanho_kernel=(21, 21)):
    
    if not os.path.isfile(caminho_imagem):
        print(f"Arquivo não encontrado: {caminho_imagem}")
        return

    imagem_bgr = cv2.imread(caminho_imagem)

    if imagem_bgr is None:
        print("Não foi possível abrir a imagem.")
        return

    imagem_rgb = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)

    cinza, desfoque, esboco = gerar_esboco(
        imagem_bgr,
        tamanho_kernel
    )

    fig, ax = plt.subplots(1, 4, figsize=(20, 5))

    ax[0].imshow(imagem_rgb)
    ax[0].set_title(f"Imagem original: {nome_imagem}")
    ax[0].axis("off")

    ax[1].imshow(cinza, cmap="gray")
    ax[1].set_title("Imagem em tons de cinza")
    ax[1].axis("off")

    ax[2].imshow(desfoque, cmap="gray")
    ax[2].set_title(f"Filtro Gaussiano {tamanho_kernel}")
    ax[2].axis("off")

    ax[3].imshow(esboco, cmap="gray")
    ax[3].set_title("Resultado: esboço")
    ax[3].axis("off")

    plt.tight_layout()
    plt.show()

In [9]:
# exibindo os resultados

mostrar_resultados(
    "imagens/bolo.jpg",
    "Bolo"
)

Arquivo não encontrado: imagens/bolo.jpg
